### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="splice",
    dataset_year="1991",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5M888",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/splice/ && wget -P local-data-warehouse/splice/ https://archive.ics.uci.edu/static/public/69/molecular+biology+splice+junction+gene+sequences.zip && unzip local-data-warehouse/splice/molecular+biology+splice+junction+gene+sequences.zip -d local-data-warehouse/splice/
""",
    # References
    academic_reference_bibtex="""@article{towell1994knowledge,
  title={Knowledge-based artificial neural networks},
  author={Towell, Geoffrey G and Shavlik, Jude W},
  journal={Artificial intelligence},
  volume={70},
  number={1-2},
  pages={119--165},
  year={1994},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="towell1994knowledge",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We tabularized the fixed-size DNA sequences from the original dataset based on the nucleotide position.
- We removed the instance ID.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SiteType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="SiteType",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/splice.data", index_col=False, header=None)

# Strip tab-like whitespaces from original data
df = df.map(lambda x: x.strip())
df = df.drop(columns=[1])  # Drop instance ID
target_feature = "SiteType"

split_data = df[2].apply(lambda x: pd.Series(list(x)))
# Generate new column names from -30 to +30 excluding 0
split_data.columns = [f"position_{i}" for i in range(-30, 31) if i != 0]
split_data[target_feature] = df[0]
df = split_data

cat_features = [f"position_{i}" for i in range(-30, 31) if i != 0] + [target_feature]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 3,190
Columns: 61
Use sampling: False (sample size: 3,190)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['position_6', 'position_5', 'position_-3', 'position_-29', 'position_-30', 'position_-9', 'position_-8', 'position_-7', 'position_-6', 'position_-12']
Rows remaining as candidates after top-10 filter: 499 (of 3,190)

#### Duplicate Report
Total duplicate rows: 184 (5.77% of dataset)
Duplicate rows ignoring target: 185 (5.80% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,position_-30,position_-29,position_-28,position_-27,position_-26,position_-25,position_-24,position_-23,position_-22,position_-21,position_-20,position_-19,position_-18,position_-17,position_-16,position_-15,position_-14,position_-13,position_-12,position_-11,position_-10,position_-9,position_-8,position_-7,position_-6,position_-5,position_-4,position_-3,position_-2,position_-1,position_1,position_2,position_3,position_4,position_5,position_6,position_7,position_8,position_9,position_10,position_11,position_12,position_13,position_14,position_15,position_16,position_17,position_18,position_19,position_20,position_21,position_22,position_23,position_24,position_25,position_26,position_27,position_28,position_29,position_30,SiteType
0,T,C,C,T,T,G,A,C,C,T,G,G,G,T,T,C,C,C,C,C,T,C,T,C,C,T,G,C,A,G,A,A,C,G,A,T,T,C,C,C,T,G,A,T,G,A,G,G,C,A,G,A,T,G,C,G,G,G,A,A,IE
1,T,T,G,A,T,A,A,C,A,T,G,A,C,A,T,T,T,T,C,C,T,T,T,T,C,T,A,C,A,G,A,A,T,G,A,A,A,C,A,G,T,A,G,A,A,G,T,C,A,T,C,T,C,A,G,A,A,A,T,G,IE
2,C,A,T,G,C,C,T,T,G,A,A,T,T,T,C,T,T,T,T,C,T,G,C,A,C,G,A,C,A,G,G,T,C,T,G,C,C,A,G,C,T,T,A,C,A,T,T,T,A,C,C,C,A,A,A,C,T,G,T,C,IE
3,G,A,T,T,C,T,C,T,T,C,A,G,C,C,A,A,T,C,T,T,C,A,T,T,G,C,T,C,A,A,G,T,A,T,G,A,C,T,T,T,A,A,T,C,T,T,C,C,T,T,A,C,A,A,C,T,A,G,G,T,EI
4,T,C,C,C,T,C,C,A,T,T,G,C,C,C,T,C,C,G,G,T,T,T,C,T,C,C,C,C,A,G,G,C,T,C,C,C,G,G,A,C,G,T,C,C,C,T,G,C,T,C,C,T,G,G,C,T,T,T,T,G,IE


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,position_-30,category,0.0,0.0,5.0,"G, C, A, T, D"
1,position_-29,category,0.0,0.0,5.0,"C, G, A, T, D"
2,position_-28,category,0.0,0.0,4.0,"C, G, T, A"
3,position_-27,category,0.0,0.0,4.0,"C, G, A, T"
4,position_-26,category,0.0,0.0,4.0,"C, T, A, G"
5,position_-25,category,0.0,0.0,4.0,"C, G, T, A"
6,position_-24,category,0.0,0.0,4.0,"C, A, G, T"
7,position_-23,category,0.0,0.0,4.0,"C, T, A, G"
8,position_-22,category,0.0,0.0,4.0,"C, T, A, G"
9,position_-21,category,0.0,0.0,4.0,"T, C, A, G"


In [6]:
# Numeric Feature Statistics
numeric_stats

'No numeric features to summarize.'

In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column       rank                    
SiteType     1        N   1655  51.88
             2       IE    768  24.08
             3       EI    767  24.04
position_-1  1        G   1822  57.12
             2        A    474  14.86
             3        T    451  14.14
             4        C    442  13.86
             5        N      1   0.03
position_-10 1        C    938  29.40
             2        T    913  28.62
             3        G    697  21.85
             4        A    640  20.06
             5        N      2   0.06
position_-11 1        T    935  29.31
             2        C    889  27.87
             3        A    693  21.72
             4        G    671  21.03
             5        N      2   0.06
position_-12 1        C    921  28.87
             2        T    899  28.18
             3        G    712  22.32
             4        A    657  20.60
             5        N      1   0.03
position_-13 1        T    906  28.40
             2        C    889  27.87
             3        G    746  23.39
             4        A    649  20.34
position_-14 1        T    911  28.56
             2        C    908  28.46
             3        G    701  21.97
             4        A    670  21.00
position_-15 1        C    949  29.75
             2        T    857  26.87
             3        A    700  21.94
             4        G    684  21.44
position_-16 1        C    959  30.06
             2        T    798  25.02
             3        G    768  24.08
             4        A    665  20.85
position_-17 1        C    898  28.15
             2        A    797  24.98
             3        T    782  24.51
             4        G    712  22.32
             5        N      1   0.03
position_-18 1        C    878  27.52
             2        T    822  25.77
             3        G    752  23.57
             4        A    738  23.13
position_-19 1        C    930  29.15
             2        G    807  25.30
             3        T    785  24.61
             4        A    668  20.94
position_-2  1        A   1613  50.56
             2        C    536  16.80
             3        G    523  16.39
             4        T    517  16.21
             5        N      1   0.03
position_-20 1        T    872  27.34
             2        C    823  25.80
             3        G    778  24.39
             4        A    717  22.48
position_-21 1        T    849  26.61
             2        C    808  25.33
             3        A    767  24.04
             4        G    766  24.01
position_-22 1        C    909  28.50
             2        T    783  24.55
             3        A    749  23.48
             4        G    749  23.48
position_-23 1        C    878  27.52
             2        T    791  24.80
             3        A    790  24.76
             4        G    731  22.92
position_-24 1        C    858  26.90
             2        A    783  24.55
             3        G    780  24.45
             4        T    769  24.11
position_-25 1        C    898  28.15
             2        G    828  25.96
             3        T    760  23.82
             4        A    704  22.07
position_-26 1        C    865  27.12
             2        T    805  25.24
             3        A    801  25.11
             4        G    719  22.54
position_-27 1        C    884  27.71
             2        G    802  25.14
             3        A    753  23.61
             4        T    751  23.54
position_-28 1        C    876  27.46
             2        G    841  26.36
             3        T    761  23.86
             4        A    712  22.32
position_-29 1        C    858  26.90
             2        G    794  24.89
             3        A    779  24.42
             4        T    758  23.76
             5        D      1   0.03
position_-3  1        C   1309  41.03
             2        A    696  21.82
             3        T    608  19.06
             4        G    576  18.06
             5        N      1   0.03
position_-30 1        G    8

In [8]:
# Target Distribution
target_df

,count,pct
SiteType,,
N,1655,51.88
IE,768,24.08
EI,767,24.04


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to splice/019d7369-b922-718a-adbf-099484b95908
019d7369-b922-718a-adbf-099484b95908
d25acd20577bcac3c95cbb8c7e84a14a91e73d76d55764f01eb4146c3618cf78
